In [66]:
import gc
import torch
import faiss 
import random
import numpy as np
from scipy.io import mmread
import torch.nn.functional as F
from torch.nn import TripletMarginLoss
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score, davies_bouldin_score

In [67]:
rawData = mmread('../scRNA.mtx')
coo_matrix = rawData.tocoo()
print(coo_matrix)
print(coo_matrix.shape)

<COOrdinate sparse matrix of dtype 'float64'
	with 86422438 stored elements and shape (34619, 27180)>
  Coords	Values
  (30, 0)	878.85046
  (44, 0)	2636.5513
  (58, 0)	878.85046
  (74, 0)	878.85046
  (77, 0)	878.85046
  (109, 0)	878.85046
  (225, 0)	3515.4019
  (247, 0)	878.85046
  (272, 0)	878.85046
  (273, 0)	1318.2756
  (365, 0)	878.85046
  (366, 0)	2636.5513
  (421, 0)	878.85046
  (431, 0)	1757.7009
  (440, 0)	878.85046
  (442, 0)	5273.1025
  (499, 0)	16698.158
  (594, 0)	878.85046
  (649, 0)	1757.7009
  (688, 0)	878.85046
  (706, 0)	878.85046
  (831, 0)	1757.7009
  (835, 0)	2636.5513
  (867, 0)	878.85046
  (911, 0)	2636.5513
  :	:
  (32162, 27179)	142.14685
  (32202, 27179)	88.841736
  (32298, 27179)	177.68347
  (32423, 27179)	59.227825
  (32425, 27179)	177.68347
  (32564, 27179)	88.841736
  (33015, 27179)	177.68347
  (33454, 27179)	177.68347
  (33504, 27179)	177.68347
  (33538, 27179)	177.68347
  (33578, 27179)	88.841736
  (33636, 27179)	177.68347
  (33647, 27179)	284.01257
  (33

In [ ]:
def clean_and_split_data(coo_matrix, max_number):
    #get only non-zero values
    total_nnz = coo_matrix.nnz 

    # Ensure max_nnz doesn’t exceed total
    if max_number >= total_nnz:
        raise ValueError(f"max_nnz ({max_number}) must be less than total non-zero elements ({total_nnz})")
    
    rows = coo_matrix.row
    cols = coo_matrix.col
    data = coo_matrix.data
    
    selected_indices = np.arange(max_number)  

    selected = coo_matrix.__class__(
        (data[selected_indices], (rows[selected_indices], cols[selected_indices])),
        shape=coo_matrix.shape
    )
    
    return selected

processed_data = clean_and_split_data(coo_matrix=coo_matrix, max_number=1000000)
print(processed_data)

<COOrdinate sparse matrix of dtype 'float64'
	with 100000 stored elements and shape (34619, 27180)>
  Coords	Values
  (30, 0)	878.85046
  (44, 0)	2636.5513
  (58, 0)	878.85046
  (74, 0)	878.85046
  (77, 0)	878.85046
  (109, 0)	878.85046
  (225, 0)	3515.4019
  (247, 0)	878.85046
  (272, 0)	878.85046
  (273, 0)	1318.2756
  (365, 0)	878.85046
  (366, 0)	2636.5513
  (421, 0)	878.85046
  (431, 0)	1757.7009
  (440, 0)	878.85046
  (442, 0)	5273.1025
  (499, 0)	16698.158
  (594, 0)	878.85046
  (649, 0)	1757.7009
  (688, 0)	878.85046
  (706, 0)	878.85046
  (831, 0)	1757.7009
  (835, 0)	2636.5513
  (867, 0)	878.85046
  (911, 0)	2636.5513
  :	:
  (23705, 27)	27.984615
  (23708, 27)	27.984615
  (23715, 27)	59.02638
  (23718, 27)	55.96923
  (23728, 27)	27.984615
  (23734, 27)	55.96923
  (23736, 27)	27.984615
  (23749, 27)	27.984615
  (23750, 27)	41.97692
  (23761, 27)	55.96923
  (23801, 27)	27.984615
  (23831, 27)	27.984615
  (23836, 27)	28.157362
  (23884, 27)	13.992308
  (23950, 27)	27.984615
  (

In [69]:
def cell_graph(data, threshold):

    gene_expression = data.data
    
    x = np.asarray(gene_expression, dtype=np.float32)
    x = x.reshape(-1, 1)


    gpu_resource_manager = faiss.StandardGpuResources() 
    similarity_object = faiss.IndexFlatL2(1)
    similarity_object_in_gpu = faiss.index_cpu_to_gpu(gpu_resource_manager, 0, similarity_object)


    print(similarity_object_in_gpu.is_trained)  
    print(f"FAISS index type: {type(similarity_object_in_gpu)}") 


    similarity_object_in_gpu.add(x)
    k=2
    distances, indices = similarity_object_in_gpu.search(x, k + 1)
    
    edge_index_list = []
    outliers = []
    
    for i in range(len(gene_expression)):
        nearest_neighbors = indices[i, 1:k+1]  
        neighbor_distances = distances[i, 1:k+1]
        
        for j, dist in zip(nearest_neighbors, neighbor_distances):
            if dist <= threshold ** 2:
                edge_index_list.append((i, j))
            else:
                outliers.append(int(j))
    

    edge_index_np = np.array(edge_index_list).T
    edge_index = torch.tensor(edge_index_np, dtype=torch.long) if edge_index_np.size > 0 else torch.empty((2, 0), dtype=torch.long)

    cleaned_outliers = list(set(outliers))
    print(cleaned_outliers)

    x_tensor = torch.tensor(x, dtype=torch.float32)
    pyg_data = Data(edge_index=edge_index, x=x_tensor)
    print(pyg_data)
    return pyg_data

data = cell_graph(data=processed_data,threshold=500)

True
FAISS index type: <class 'faiss.swigfaiss.GpuIndexFlat'>
[76675, 55779, 9220, 66923, 75244, 75245, 75408, 43504, 24500, 72277, 36222]
Data(x=[100000, 1], edge_index=[2, 199979])


In [70]:
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers):
        super(GraphSAGE, self).__init__()
        self.convs = torch.nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden_channels))  
        
        for _ in range(num_layers - 2):  
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        
        self.convs.append(SAGEConv(hidden_channels, out_channels))  

    def forward(self, x, edge_index):
        for conv in self.convs[:-1]: 
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=0.5, training=self.training)
        x = self.convs[-1](x, edge_index) 
        return x  

In [71]:
model1 = torch.load("./save_trained_models/entire_model1.pth")
model3 = torch.load("./save_trained_models/entire_model3.pth")
model4 = torch.load("./save_trained_models/entire_model4.pth")

C:\Users\shera\AppData\Local\Temp\ipykernel_30536\3102131071.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1 = torch.load("./save_trained_models/entire_model1.pth

In [72]:
model1 = model1.to('cuda')
model1.eval()

# Send input data to GPU
x = data.x.to('cuda')
edge_index = data.edge_index.to('cuda')

# Get embeddings
with torch.no_grad():
    embeddings_model1 = model1(x, edge_index)

print(embeddings_model1.shape)

torch.Size([100000, 64])


In [73]:
embeddings_np = embeddings_model1.cpu().numpy()

d = embeddings_np.shape[1]  
k = 10
kmeans = faiss.Kmeans(d, k, niter=300, gpu=True)  


kmeans.train(embeddings_np)
_, labels = kmeans.index.search(embeddings_np, 1)


labels = torch.tensor(labels.flatten())
print(labels.shape)

torch.Size([100000])


In [74]:
embeddings_cpu = embeddings_model1.cpu().detach().numpy()

# Now you can calculate the Silhouette Score
sil_score = silhouette_score(embeddings_cpu, labels)
print(f"Silhouette Score: {sil_score:.4f}")

# Davies-Bouldin Index (lower is better)
db_index = davies_bouldin_score(embeddings_cpu, labels)
print(f"Davies-Bouldin Index: {db_index:.4f}")

Silhouette Score: 0.7538
Davies-Bouldin Index: 0.7733


In [75]:
model3 = model3.to('cuda')
model3.eval()

# Send input data to GPU
x = data.x.to('cuda')
edge_index = data.edge_index.to('cuda')

# Get embeddings
with torch.no_grad():
    embeddings_model3 = model3(x, edge_index)

print(embeddings_model3.shape)

torch.Size([100000, 4])


In [76]:
embeddings_np = embeddings_model3.cpu().numpy()

d = embeddings_np.shape[1]  
k = 10
kmeans = faiss.Kmeans(d, k, niter=300, gpu=True)  


kmeans.train(embeddings_np)
_, labels = kmeans.index.search(embeddings_np, 1)


labels = torch.tensor(labels.flatten())
print(labels.shape)

torch.Size([100000])


In [77]:
embeddings_cpu = embeddings_model3.cpu().numpy()

# Now you can calculate the Silhouette Score
sil_score = silhouette_score(embeddings_cpu, labels)
print(f"Silhouette Score: {sil_score:.4f}")

# Davies-Bouldin Index (lower is better)
db_index = davies_bouldin_score(embeddings_cpu, labels)
print(f"Davies-Bouldin Index: {db_index:.4f}")

Silhouette Score: 0.7097
Davies-Bouldin Index: 0.7610


In [78]:
model4 = model4.to('cuda')
model4.eval()

# Send input data to GPU
x = data.x.to('cuda')
edge_index = data.edge_index.to('cuda')

# Get embeddings
with torch.no_grad():
    embeddings_model4 = model4(x, edge_index)

print(embeddings_model4.shape)

torch.Size([100000, 4])


In [79]:
embeddings_np = embeddings_model4.cpu().numpy()

d = embeddings_np.shape[1]  
k = 10
kmeans = faiss.Kmeans(d, k, niter=300, gpu=True)  


kmeans.train(embeddings_np)
_, labels = kmeans.index.search(embeddings_np, 1)


labels = torch.tensor(labels.flatten())
print(labels.shape)

torch.Size([100000])


In [80]:
embeddings_cpu = embeddings_model4.cpu().numpy()

# Now you can calculate the Silhouette Score
sil_score = silhouette_score(embeddings_cpu, labels)
print(f"Silhouette Score: {sil_score:.4f}")

# Davies-Bouldin Index (lower is better)
db_index = davies_bouldin_score(embeddings_cpu, labels)
print(f"Davies-Bouldin Index: {db_index:.4f}")

Silhouette Score: 0.6114
Davies-Bouldin Index: 0.9523
